# Feedback integration: Tables & Figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
from scipy import stats

matplotlib.rcParams.update({'font.size': 10, 'figure.dpi': 150})

RAW = '../results/raw_csv'
TABLES = '../results/tables'
FIGURES = '../results/figures'

master = pd.read_csv(f'{RAW}/master_analysis.csv')
tests = pd.read_csv(f'{RAW}/tests_all.csv')
cppcheck_det = pd.read_csv(f'{RAW}/cppcheck_detailed.csv')
cppcheck_sum = pd.read_csv(f'{RAW}/cppcheck_summary.csv')
clang_sum = pd.read_csv(f'{RAW}/clang_tidy_summary.csv')
clang_det = pd.read_csv(f'{RAW}/clang_tidy_detailed.csv')
agg = pd.read_csv(f'{RAW}/agg_per_model_gen.csv')

print(f'master: {len(master)} rows')
print(f'tests_all: {len(tests)} rows')
print(f'cppcheck_detailed: {len(cppcheck_det)} rows')
print(f'clang_tidy_detailed: {len(clang_det)} rows')

---
## Q2: Compilation & Runtime Success Rates per Model/Generation

> "How do you handle compilation failures and runtime errors in Test Pass Rate computation? Are compile failures counted as zero pass rate, or excluded? Please report compile/run success rates per model/generation."

In [ ]:
# Pipeline attrition table: total generated → compiled → had tests → passed ≥1 → all passed
q2 = tests.groupby(['model', 'generation']).agg(
    total_generated=('sample_id', 'count'),
    compiled=('compiled', 'sum'),
    had_tests=('has_tests', 'sum'),
    timed_out=('timeout', 'sum'),
).reset_index()

# Compute rates
q2['compile_rate'] = (q2['compiled'] / q2['total_generated']).round(3)

# For TPR: only programs with has_tests=1 are included; compile failures with has_tests=0 are excluded
# Programs that compiled but have no tests get pass_rate=NaN (excluded from TPR)
# Programs that failed to compile get compiled=0, pass_rate=NaN (excluded)
tpr_data = tests[tests['has_tests'] == 1].copy()
tpr_stats = tpr_data.groupby(['model', 'generation']).agg(
    n_with_tests=('sample_id', 'count'),
    mean_tpr=('pass_rate', 'mean'),
    all_pass=('pass_rate', lambda x: (x == 1.0).sum()),
    zero_pass=('pass_rate', lambda x: (x == 0.0).sum()),
).reset_index()

q2_full = q2.merge(tpr_stats, on=['model', 'generation'])
q2_full['test_coverage'] = (q2_full['had_tests'] / q2_full['total_generated']).round(3)
q2_full['timeout_rate'] = (q2_full['timed_out'] / q2_full['n_with_tests']).round(3)
q2_full['mean_tpr'] = q2_full['mean_tpr'].round(3)

cols = ['model', 'generation', 'total_generated', 'compiled', 'compile_rate',
        'had_tests', 'test_coverage', 'n_with_tests', 'mean_tpr',
        'all_pass', 'zero_pass', 'timed_out', 'timeout_rate']
q2_display = q2_full[cols].copy()
q2_display.columns = ['Model', 'Generation', 'Total Generated', 'Compiled', 'Compile Rate',
                       'Had Tests', 'Test Coverage', 'N (TPR calc)', 'Mean TPR',
                       'All Pass', 'Zero Pass', 'Timed Out', 'Timeout Rate']

q2_display.to_csv(f'{TABLES}/q2_pipeline_attrition.csv', index=False)
print('Table Q2: Pipeline Attrition per Model/Generation')
q2_display

In [ ]:
# Figure Q2: Stacked bar chart of pipeline attrition
fig, ax = plt.subplots(figsize=(12, 5))

ai_only = q2_full[~q2_full['model'].isin(['human'])].copy()
ai_only['label'] = ai_only['model'] + '/' + ai_only['generation']
ai_only = ai_only.sort_values(['model', 'generation'])

x = range(len(ai_only))
w = 0.6

not_compiled = ai_only['total_generated'] - ai_only['compiled']
compiled_no_tests = ai_only['compiled'] - ai_only['had_tests']
had_tests_not_pass = ai_only['n_with_tests'] - ai_only['all_pass']
all_passed = ai_only['all_pass']

ax.bar(x, not_compiled.values, w, label='Failed to compile', color='#d62728')
ax.bar(x, compiled_no_tests.values, w, bottom=not_compiled.values, label='Compiled, no tests', color='#ff7f0e')
ax.bar(x, had_tests_not_pass.values, w, bottom=(not_compiled + compiled_no_tests).values, label='Had tests, not all passed', color='#1f77b4')
ax.bar(x, all_passed.values, w, bottom=(not_compiled + compiled_no_tests + had_tests_not_pass).values, label='All tests passed', color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(ai_only['label'].values, rotation=45, ha='right')
ax.set_ylabel('Number of Programs')
ax.set_title('Pipeline Attrition: From Generation to All Tests Passed (AI Models)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIGURES}/q2_pipeline_attrition.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q2_pipeline_attrition.png', bbox_inches='tight')
plt.show()

---
## Q4: Extraction Exclusion Rates (Fenced Code Blocks)

> "What fraction of generations lacked fenced C++ code blocks and were therefore excluded? Could this exclusion bias both TPR and static analysis prevalence?"

In [ ]:
# 851 total instructions. Per model/gen, count how many were actually generated
# The difference (851 - generated) = failed API calls or responses without valid code

total_instructions = 851

q4_data = []
for _, row in agg.iterrows():
    generated = row['total_files']
    excluded = total_instructions - generated
    q4_data.append({
        'Model': row['model'],
        'Generation': row['gen'],
        'Total Instructions': total_instructions,
        'Extracted Programs': int(generated),
        'Excluded (no code / API fail)': int(excluded),
        'Exclusion Rate': round(excluded / total_instructions, 3),
        'Compiled': int(row['compiled_count']),
        'Compile Rate (of extracted)': round(row['compile_rate'], 3)
    })

q4_df = pd.DataFrame(q4_data)
q4_df.to_csv(f'{TABLES}/q4_extraction_exclusion.csv', index=False)
print('Table Q4: Extraction and Exclusion Rates')
q4_df

---
## Q5: Per-CWE Distributions (Top 10 by Frequency) with Severity

> "For RQ1, can you provide per-CWE distributions (top 10 by frequency) and severity aggregation, rather than only 'any CWE' prevalence?"

In [ ]:
# MITRE CWE severity mapping (based on MITRE Top-25 2024 and CVSS typical scores)
cwe_severity = {
    'CWE-119': ('Buffer Overflow', 'High', True),
    'CWE-190': ('Integer Overflow', 'High', True),
    'CWE-197': ('Numeric Truncation', 'Medium', False),
    'CWE-252': ('Unchecked Return Value', 'Medium', False),
    'CWE-369': ('Divide By Zero', 'Medium', False),
    'CWE-391': ('Unchecked Error Condition', 'Low', False),
    'CWE-398': ('Code Quality (Variable Scope)', 'Low', False),
    'CWE-457': ('Use of Uninitialized Variable', 'High', True),
    'CWE-476': ('NULL Pointer Dereference', 'High', True),
    'CWE-477': ('Obsolete Function', 'Low', False),
    'CWE-561': ('Dead Code', 'Low', False),
    'CWE-562': ('Return of Stack Variable Address', 'High', False),
    'CWE-563': ('Unused Variable Assignment', 'Low', False),
    'CWE-570': ('Expression Always False', 'Medium', False),
    'CWE-571': ('Expression Always True', 'Medium', False),
    'CWE-628': ('Function Call Wrong Args', 'Medium', False),
    'CWE-664': ('Improper Resource Control', 'Medium', False),
    'CWE-665': ('Improper Initialization', 'Medium', False),
    'CWE-686': ('Function Call Wrong Arg Type', 'Medium', False),
    'CWE-704': ('Incorrect Type Conversion', 'Medium', False),
    'CWE-758': ('Undefined Behavior', 'High', False),
    'CWE-762': ('Mismatched Memory Routines', 'High', False),
    'CWE-772': ('Missing Resource Release', 'Medium', False),
    'CWE-783': ('Operator Precedence Error', 'Medium', False),
    'CWE-786': ('Buffer Under-read', 'High', True),
    'CWE-788': ('Buffer Over-read', 'High', True),
    'CWE-825': ('Expired Pointer Dereference', 'High', False),
}

# Build per-CWE table with total findings across all models
cwe_prev = pd.read_csv(f'{RAW}/cwe_prevalence.csv')

# Aggregate findings per CWE across AI models only
ai_cwes = cwe_prev[cwe_prev['model'] != 'human'].groupby('cwe').agg(
    ai_total_findings=('total_findings', 'sum'),
    ai_files_affected=('files_affected', 'sum')
).reset_index()

human_cwes = cwe_prev[cwe_prev['model'] == 'human'].groupby('cwe').agg(
    human_total_findings=('total_findings', 'sum'),
    human_files_affected=('files_affected', 'sum')
).reset_index()

q5 = ai_cwes.merge(human_cwes, on='cwe', how='outer').fillna(0)

# Add severity info
q5['Description'] = q5['cwe'].map(lambda x: cwe_severity.get(x, ('Unknown', 'Unknown', False))[0])
q5['Severity'] = q5['cwe'].map(lambda x: cwe_severity.get(x, ('Unknown', 'Unknown', False))[1])
q5['MITRE Top-25'] = q5['cwe'].map(lambda x: cwe_severity.get(x, ('Unknown', 'Unknown', False))[2])

q5 = q5.sort_values('ai_total_findings', ascending=False)
q5_top10 = q5.head(10).copy()
q5_top10.columns = ['CWE', 'AI Total Findings', 'AI Files Affected',
                     'Human Total Findings', 'Human Files Affected',
                     'Description', 'Severity', 'MITRE Top-25']

q5_top10 = q5_top10[['CWE', 'Description', 'Severity', 'MITRE Top-25',
                      'AI Total Findings', 'AI Files Affected',
                      'Human Total Findings', 'Human Files Affected']]
q5_top10['AI Total Findings'] = q5_top10['AI Total Findings'].astype(int)
q5_top10['AI Files Affected'] = q5_top10['AI Files Affected'].astype(int)
q5_top10['Human Total Findings'] = q5_top10['Human Total Findings'].astype(int)
q5_top10['Human Files Affected'] = q5_top10['Human Files Affected'].astype(int)

q5_top10.to_csv(f'{TABLES}/q5_top10_cwe_with_severity.csv', index=False)
print('Table Q5: Top 10 CWEs by Frequency with Severity')
q5_top10

In [ ]:
# Per-model breakdown of top-10 CWEs
top10_cwes = q5_top10['CWE'].tolist()
cwe_by_model = cwe_prev[cwe_prev['cwe'].isin(top10_cwes)].copy()

# Pivot: rows=CWE, columns=model, values=total_findings
q5_pivot = cwe_by_model.pivot_table(
    index='cwe', columns='model', values='total_findings', fill_value=0
).astype(int)

# Add description/severity
q5_pivot['Description'] = q5_pivot.index.map(lambda x: cwe_severity.get(x, ('?','?',False))[0])
q5_pivot['Severity'] = q5_pivot.index.map(lambda x: cwe_severity.get(x, ('?','?',False))[1])

q5_pivot = q5_pivot[['Description', 'Severity', 'gemma', 'llama', 'qwen', 'human']]
q5_pivot = q5_pivot.sort_values('gemma', ascending=False)

q5_pivot.to_csv(f'{TABLES}/q5_cwe_by_model_with_severity.csv')
print('Table Q5b: CWE Findings by Model with Severity')
q5_pivot

In [ ]:
# Figure Q5: Grouped bar chart of top-10 CWEs by model
fig, ax = plt.subplots(figsize=(14, 6))

models = ['gemma', 'llama', 'qwen', 'human']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
x = np.arange(len(top10_cwes))
width = 0.2

for i, (model, color) in enumerate(zip(models, colors)):
    vals = []
    for cwe in top10_cwes:
        row = cwe_by_model[(cwe_by_model['cwe'] == cwe) & (cwe_by_model['model'] == model)]
        vals.append(row['total_findings'].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, vals, width, label=model.capitalize(), color=color)

# Color x-tick labels by severity
sev_colors = {'High': 'red', 'Medium': 'orange', 'Low': 'gray'}
labels = [f"{c}\n({cwe_severity.get(c, ('?','?',False))[0][:15]})" for c in top10_cwes]

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)

# Color labels by severity
for i, cwe in enumerate(top10_cwes):
    sev = cwe_severity.get(cwe, ('?','?',False))[1]
    ax.get_xticklabels()[i].set_color(sev_colors.get(sev, 'black'))

ax.set_ylabel('Total Findings')
ax.set_title('Top 10 CWE Types by Frequency (colored labels = severity: red=High, orange=Medium, gray=Low)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}/q5_top10_cwe_by_model_severity.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q5_top10_cwe_by_model_severity.png', bbox_inches='tight')
plt.show()

In [ ]:
# Severity aggregation table
all_cwes_with_sev = q5.copy()
sev_agg = all_cwes_with_sev.groupby('Severity').agg(
    n_cwe_types=('cwe', 'count'),
    ai_total=('ai_total_findings', 'sum'),
    ai_files=('ai_files_affected', 'sum'),
    human_total=('human_total_findings', 'sum'),
    human_files=('human_files_affected', 'sum')
).reset_index()

sev_agg.columns = ['Severity', '# CWE Types', 'AI Total Findings', 'AI Files',
                    'Human Total Findings', 'Human Files']
sev_agg = sev_agg.sort_values('AI Total Findings', ascending=False)
sev_agg.to_csv(f'{TABLES}/q5_severity_aggregation.csv', index=False)
print('Table Q5c: Severity Aggregation')
sev_agg

---
## Q8: Per-Problem Paired Comparisons (LLM vs Human)

> "How were human solutions selected and normalized? Can you provide per-problem paired comparisons (LLM vs. human) on warning counts and categories?"

In [ ]:
# Per-problem paired comparison: for each problem, average the human warning counts
# and the AI warning counts, then compare

# Human solutions
human_df = master[master['model'] == 'human'].copy()
human_agg = human_df.groupby('problem_key').agg(
    human_cppcheck_total=('cppcheck_total', 'mean'),
    human_cppcheck_cwe=('cppcheck_cwe_count', 'mean'),
    human_clang_total=('clang_total', 'mean'),
    human_clang_bugprone=('clang_bugprone', 'mean'),
    human_clang_analyzer=('clang_analyzer', 'mean'),
    human_pass_rate=('pass_rate', 'mean'),
    human_n=('sample_id', 'count')
).reset_index()

# AI solutions (average across all models and generations per problem)
ai_df = master[master['model'] != 'human'].copy()
ai_agg = ai_df.groupby('problem_key').agg(
    ai_cppcheck_total=('cppcheck_total', 'mean'),
    ai_cppcheck_cwe=('cppcheck_cwe_count', 'mean'),
    ai_clang_total=('clang_total', 'mean'),
    ai_clang_bugprone=('clang_bugprone', 'mean'),
    ai_clang_analyzer=('clang_analyzer', 'mean'),
    ai_pass_rate=('pass_rate', 'mean'),
    ai_n=('sample_id', 'count')
).reset_index()

# Also per-model
ai_per_model = ai_df.groupby(['model', 'problem_key']).agg(
    ai_cppcheck_total=('cppcheck_total', 'mean'),
    ai_cppcheck_cwe=('cppcheck_cwe_count', 'mean'),
    ai_clang_total=('clang_total', 'mean'),
    ai_clang_bugprone=('clang_bugprone', 'mean'),
    ai_clang_analyzer=('clang_analyzer', 'mean'),
    ai_pass_rate=('pass_rate', 'mean'),
).reset_index()

paired = human_agg.merge(ai_agg, on='problem_key', how='inner')
print(f'Paired problems (both human and AI): {len(paired)}')

# Summary statistics of the paired differences
metrics = [
    ('cppcheck_total', 'Cppcheck Total Warnings'),
    ('cppcheck_cwe', 'Cppcheck CWE Count'),
    ('clang_total', 'Clang-Tidy Total Diagnostics'),
    ('clang_bugprone', 'Clang-Tidy Bugprone'),
    ('clang_analyzer', 'Clang-Tidy Analyzer'),
]

paired_stats = []
for metric, label in metrics:
    h = paired[f'human_{metric}']
    a = paired[f'ai_{metric}']
    diff = h - a  # positive = human has more
    t_stat, p_val = stats.ttest_rel(h, a)
    paired_stats.append({
        'Metric': label,
        'Human Mean': round(h.mean(), 3),
        'AI Mean': round(a.mean(), 3),
        'Mean Diff (H-AI)': round(diff.mean(), 3),
        'Std Diff': round(diff.std(), 3),
        't-statistic': round(t_stat, 3),
        'p-value': f'{p_val:.2e}',
        'N problems': len(paired)
    })

q8_table = pd.DataFrame(paired_stats)
q8_table.to_csv(f'{TABLES}/q8_paired_comparison.csv', index=False)
print('Table Q8: Paired Per-Problem Comparison (Human vs AI)')
q8_table

In [ ]:
# Per-model paired comparison
paired_by_model = []
for model in ['gemma', 'llama', 'qwen']:
    model_ai = ai_per_model[ai_per_model['model'] == model]
    p = human_agg.merge(model_ai, on='problem_key', how='inner', suffixes=('', '_m'))
    for metric, label in metrics:
        h = p[f'human_{metric}']
        a = p[f'ai_{metric}']
        diff = h - a
        t_stat, p_val = stats.ttest_rel(h, a)
        paired_by_model.append({
            'Model': model.capitalize(),
            'Metric': label,
            'Human Mean': round(h.mean(), 3),
            f'{model.capitalize()} Mean': round(a.mean(), 3),
            'Mean Diff': round(diff.mean(), 3),
            'p-value': f'{p_val:.2e}',
            'N': len(p)
        })

q8b = pd.DataFrame(paired_by_model)
q8b.to_csv(f'{TABLES}/q8_paired_by_model.csv', index=False)
print('Table Q8b: Paired Comparison by Model')
q8b

In [ ]:
# Figure Q8: Scatter plot of per-problem warning counts (Human vs AI)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scatter_metrics = [
    ('cppcheck_total', 'Cppcheck Total'),
    ('clang_total', 'Clang-Tidy Total'),
    ('clang_bugprone', 'Clang-Tidy Bugprone'),
]

for ax, (metric, label) in zip(axes, scatter_metrics):
    h = paired[f'human_{metric}']
    a = paired[f'ai_{metric}']
    ax.scatter(h, a, alpha=0.3, s=15, color='steelblue')
    max_val = max(h.max(), a.max())
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
    r, p = stats.pearsonr(h, a)
    ax.set_xlabel(f'Human {label}')
    ax.set_ylabel(f'AI {label}')
    ax.set_title(f'{label}\n(r={r:.3f}, p={p:.2e})')
    ax.legend()

plt.suptitle('Per-Problem Paired Comparison: Human vs AI Warning Counts', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES}/q8_paired_scatter.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q8_paired_scatter.png', bbox_inches='tight')
plt.show()

---
## Q9: Jaccard Similarity of Warning Sets Across Generations

> "Did you analyze the stability of warning sets across generations per task (e.g., Jaccard similarity)?"

In [ ]:
# Compute per-task Jaccard similarity of cppcheck warning sets across generations
# For each (model, problem_key), compare the SET of CWE types found in gen_1 vs gen_2, gen_1 vs gen_3, gen_2 vs gen_3

def jaccard(set1, set2):
    if len(set1) == 0 and len(set2) == 0:
        return 1.0  # both empty = identical
    return len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0.0

# CWE sets per (model, generation, problem_key)
cwe_sets = cppcheck_det[cppcheck_det['model'] != 'human'].groupby(
    ['model', 'generation', 'problem_key']
)['cwe'].apply(lambda x: set(x.dropna().astype(str))).reset_index()
cwe_sets.columns = ['model', 'generation', 'problem_key', 'cwe_set']

# Clang-tidy category sets per (model, generation, problem_key)
clang_cat_sets = clang_det[clang_det['model'] != 'human'].groupby(
    ['model', 'gen', 'problem_key']
)['category'].apply(lambda x: set(x.dropna())).reset_index()
clang_cat_sets.columns = ['model', 'generation', 'problem_key', 'clang_cat_set']

# Clang-tidy diagnostic name sets
clang_diag_sets = clang_det[clang_det['model'] != 'human'].groupby(
    ['model', 'gen', 'problem_key']
)['diagnostic_name'].apply(lambda x: set(x.dropna())).reset_index()
clang_diag_sets.columns = ['model', 'generation', 'problem_key', 'clang_diag_set']

gen_pairs = [('gen_1', 'gen_2'), ('gen_1', 'gen_3'), ('gen_2', 'gen_3')]

jaccard_results = []
for model in ['gemma', 'llama', 'qwen']:
    for g1, g2 in gen_pairs:
        # CWE Jaccard
        s1 = cwe_sets[(cwe_sets['model'] == model) & (cwe_sets['generation'] == g1)]
        s2 = cwe_sets[(cwe_sets['model'] == model) & (cwe_sets['generation'] == g2)]
        merged = s1.merge(s2, on='problem_key', suffixes=('_1', '_2'))
        if len(merged) > 0:
            cwe_jaccards = merged.apply(lambda r: jaccard(r['cwe_set_1'], r['cwe_set_2']), axis=1)
        else:
            cwe_jaccards = pd.Series([0])

        # Clang category Jaccard
        c1 = clang_cat_sets[(clang_cat_sets['model'] == model) & (clang_cat_sets['generation'] == g1)]
        c2 = clang_cat_sets[(clang_cat_sets['model'] == model) & (clang_cat_sets['generation'] == g2)]
        c_merged = c1.merge(c2, on='problem_key', suffixes=('_1', '_2'))
        if len(c_merged) > 0:
            clang_cat_jaccards = c_merged.apply(lambda r: jaccard(r['clang_cat_set_1'], r['clang_cat_set_2']), axis=1)
        else:
            clang_cat_jaccards = pd.Series([0])

        # Clang diagnostic Jaccard
        d1 = clang_diag_sets[(clang_diag_sets['model'] == model) & (clang_diag_sets['generation'] == g1)]
        d2 = clang_diag_sets[(clang_diag_sets['model'] == model) & (clang_diag_sets['generation'] == g2)]
        d_merged = d1.merge(d2, on='problem_key', suffixes=('_1', '_2'))
        if len(d_merged) > 0:
            clang_diag_jaccards = d_merged.apply(lambda r: jaccard(r['clang_diag_set_1'], r['clang_diag_set_2']), axis=1)
        else:
            clang_diag_jaccards = pd.Series([0])

        jaccard_results.append({
            'Model': model.capitalize(),
            'Gen Pair': f'{g1} vs {g2}',
            'CWE Jaccard (mean)': round(cwe_jaccards.mean(), 3),
            'CWE Jaccard (median)': round(cwe_jaccards.median(), 3),
            'N problems (CWE)': len(merged),
            'Clang Category Jaccard (mean)': round(clang_cat_jaccards.mean(), 3),
            'Clang Diagnostic Jaccard (mean)': round(clang_diag_jaccards.mean(), 3),
            'N problems (Clang)': len(c_merged),
        })

q9_table = pd.DataFrame(jaccard_results)
q9_table.to_csv(f'{TABLES}/q9_jaccard_warning_stability.csv', index=False)
print('Table Q9: Per-Task Jaccard Similarity of Warning Sets Across Generations')
q9_table

In [ ]:
# Figure Q9: Jaccard similarity distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, model in enumerate(['gemma', 'llama', 'qwen']):
    ax = axes[idx]
    for g1, g2 in gen_pairs:
        s1 = cwe_sets[(cwe_sets['model'] == model) & (cwe_sets['generation'] == g1)]
        s2 = cwe_sets[(cwe_sets['model'] == model) & (cwe_sets['generation'] == g2)]
        merged = s1.merge(s2, on='problem_key', suffixes=('_1', '_2'))
        if len(merged) > 0:
            jacs = merged.apply(lambda r: jaccard(r['cwe_set_1'], r['cwe_set_2']), axis=1)
            ax.hist(jacs, bins=20, alpha=0.5, label=f'{g1} vs {g2}', density=True)
    ax.set_xlabel('Jaccard Similarity')
    ax.set_ylabel('Density')
    ax.set_title(f'{model.capitalize()}: CWE Set Stability')
    ax.legend(fontsize=8)

plt.suptitle('Per-Task CWE Set Jaccard Similarity Across Generations', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES}/q9_jaccard_cwe_distributions.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q9_jaccard_cwe_distributions.png', bbox_inches='tight')
plt.show()

---
## Q11: Sensitivity Analysis (Excluding Inconclusive + Adding Clang-Tidy)

> "Do your conclusions change if inconclusive cppcheck warnings are excluded or if a second analyzer (e.g., clang-tidy) is added? Please provide a sensitivity analysis."

In [ ]:
# Sensitivity analysis 1: CWE prevalence with vs without style/inconclusive cppcheck findings
# "Inconclusive" in cppcheck = style-level warnings (CWE-398 variableScope, CWE-563 unreadVariable)

# Full cppcheck: any CWE
full_cwe = master.groupby(['model', 'generation']).agg(
    n_analyzed=('sample_id', 'count'),
    n_with_cwe=('cppcheck_has_cwe', 'sum')
).reset_index()
full_cwe['prevalence_full'] = (full_cwe['n_with_cwe'] / full_cwe['n_analyzed']).round(3)

# Excluding style-only CWEs (CWE-398 and CWE-563)
style_cwes = {'398', '563', '398.0', '563.0'}

# Get programs with non-style CWEs from detailed data
non_style_cwe = cppcheck_det[~cppcheck_det['cwe'].astype(str).isin(style_cwes)].copy()
non_style_programs = non_style_cwe.groupby(['model', 'generation', 'sample_id']).size().reset_index()
non_style_programs.columns = ['model', 'generation', 'sample_id', 'n_findings']

non_style_by_mg = non_style_programs.groupby(['model', 'generation'])['sample_id'].nunique().reset_index()
non_style_by_mg.columns = ['model', 'generation', 'n_with_non_style_cwe']

sensitivity = full_cwe.merge(non_style_by_mg, on=['model', 'generation'], how='left')
sensitivity['n_with_non_style_cwe'] = sensitivity['n_with_non_style_cwe'].fillna(0).astype(int)
sensitivity['prevalence_excl_style'] = (sensitivity['n_with_non_style_cwe'] / sensitivity['n_analyzed']).round(3)

# Clang-tidy security prevalence (bugprone + analyzer > 0)
master['clang_security'] = master['clang_bugprone'].fillna(0) + master['clang_analyzer'].fillna(0)
clang_sec = master.groupby(['model', 'generation']).agg(
    n_with_clang_security=('clang_security', lambda x: (x > 0).sum())
).reset_index()

sensitivity = sensitivity.merge(clang_sec, on=['model', 'generation'], how='left')
sensitivity['prevalence_clang_security'] = (sensitivity['n_with_clang_security'] / sensitivity['n_analyzed']).round(3)

# Display
q11_cols = ['model', 'generation', 'n_analyzed', 'n_with_cwe', 'prevalence_full',
            'n_with_non_style_cwe', 'prevalence_excl_style',
            'n_with_clang_security', 'prevalence_clang_security']
q11 = sensitivity[q11_cols].copy()
q11.columns = ['Model', 'Generation', 'N Analyzed', 'N w/ Any CWE', 'Prevalence (all)',
               'N w/ Non-Style CWE', 'Prevalence (excl. style)',
               'N w/ Clang Security', 'Prevalence (clang security)']

q11.to_csv(f'{TABLES}/q11_sensitivity_analysis.csv', index=False)
print('Table Q11: Sensitivity Analysis — CWE Prevalence under Different Criteria')
q11

In [ ]:
# Figure Q11: Side-by-side comparison of prevalence under different criteria
fig, ax = plt.subplots(figsize=(14, 6))

ai_sens = q11[~q11['Model'].isin(['human'])].copy()
ai_sens['label'] = ai_sens['Model'] + '/' + ai_sens['Generation']

x = np.arange(len(ai_sens))
w = 0.25

ax.bar(x - w, ai_sens['Prevalence (all)'].values, w, label='Cppcheck (all CWEs)', color='#1f77b4')
ax.bar(x, ai_sens['Prevalence (excl. style)'].values, w, label='Cppcheck (excl. style CWEs)', color='#ff7f0e')
ax.bar(x + w, ai_sens['Prevalence (clang security)'].values, w, label='Clang-Tidy (bugprone+analyzer)', color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(ai_sens['label'].values, rotation=45, ha='right')
ax.set_ylabel('Fraction of Programs with ≥1 Finding')
ax.set_title('Sensitivity Analysis: Vulnerability Prevalence by Tool and Criteria')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(f'{FIGURES}/q11_sensitivity_analysis.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q11_sensitivity_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Tool agreement analysis: programs flagged by cppcheck vs clang-tidy
master['has_cppcheck_cwe'] = master['cppcheck_has_cwe'].fillna(0).astype(bool)
master['has_clang_security'] = (master['clang_security'] > 0)

tool_agreement = []
for model in ['gemma', 'llama', 'qwen', 'human']:
    m = master[master['model'] == model]
    both = ((m['has_cppcheck_cwe']) & (m['has_clang_security'])).sum()
    cpp_only = ((m['has_cppcheck_cwe']) & (~m['has_clang_security'])).sum()
    clang_only = ((~m['has_cppcheck_cwe']) & (m['has_clang_security'])).sum()
    neither = ((~m['has_cppcheck_cwe']) & (~m['has_clang_security'])).sum()
    total = len(m)
    tool_agreement.append({
        'Model': model.capitalize(),
        'Both tools': both,
        'Cppcheck only': cpp_only,
        'Clang-tidy only': clang_only,
        'Neither': neither,
        'Total': total,
        'Agreement rate': round((both + neither) / total, 3)
    })

q11_agree = pd.DataFrame(tool_agreement)
q11_agree.to_csv(f'{TABLES}/q11_tool_agreement.csv', index=False)
print('Table Q11b: Tool Agreement (Cppcheck CWE vs Clang-Tidy Security)')
q11_agree

---
## Q12: Correlation Between TPR and Static Warning Presence

> "Did you examine correlations between test pass rate and static warning presence at the program level? Are correct solutions systematically 'cleaner,' or is there no relationship?"

In [ ]:
# Program-level correlation: pass_rate vs. warning counts
# Only include programs with test results (pass_rate is not NaN)
tested = master[master['pass_rate'].notna()].copy()

corr_metrics = [
    ('cppcheck_total', 'Cppcheck Total'),
    ('cppcheck_cwe_count', 'Cppcheck CWE Count'),
    ('clang_total', 'Clang-Tidy Total'),
    ('clang_bugprone', 'Clang-Tidy Bugprone'),
    ('clang_analyzer', 'Clang-Tidy Analyzer'),
]

# Overall correlation
corr_results = []
for col, label in corr_metrics:
    valid = tested[['pass_rate', col]].dropna()
    r_pearson, p_pearson = stats.pearsonr(valid['pass_rate'], valid[col])
    r_spearman, p_spearman = stats.spearmanr(valid['pass_rate'], valid[col])
    corr_results.append({
        'Metric': label,
        'Pearson r': round(r_pearson, 4),
        'Pearson p': f'{p_pearson:.2e}',
        'Spearman rho': round(r_spearman, 4),
        'Spearman p': f'{p_spearman:.2e}',
        'N': len(valid)
    })

q12_overall = pd.DataFrame(corr_results)
q12_overall.to_csv(f'{TABLES}/q12_tpr_warning_correlation_overall.csv', index=False)
print('Table Q12: Overall Correlation Between TPR and Static Warning Counts')
q12_overall

In [ ]:
# Per-model correlation
per_model_corr = []
for model in ['gemma', 'llama', 'qwen', 'human']:
    m = tested[tested['model'] == model]
    for col, label in corr_metrics:
        valid = m[['pass_rate', col]].dropna()
        if len(valid) > 2:
            r, p = stats.spearmanr(valid['pass_rate'], valid[col])
            per_model_corr.append({
                'Model': model.capitalize(),
                'Metric': label,
                'Spearman rho': round(r, 4),
                'p-value': f'{p:.2e}',
                'N': len(valid)
            })

q12_by_model = pd.DataFrame(per_model_corr)
q12_by_model.to_csv(f'{TABLES}/q12_tpr_warning_correlation_by_model.csv', index=False)
print('Table Q12b: Per-Model Spearman Correlation Between TPR and Static Warnings')
q12_by_model

In [ ]:
# Compare mean warning counts: passing vs failing programs
tested['status'] = tested['pass_rate'].apply(
    lambda x: 'all_pass' if x == 1.0 else ('partial' if x > 0 else 'zero_pass')
)

status_comparison = []
for model in ['gemma', 'llama', 'qwen', 'human']:
    m = tested[tested['model'] == model]
    for status in ['all_pass', 'partial', 'zero_pass']:
        s = m[m['status'] == status]
        if len(s) > 0:
            status_comparison.append({
                'Model': model.capitalize(),
                'Status': status,
                'N': len(s),
                'Mean Cppcheck Total': round(s['cppcheck_total'].mean(), 3),
                'Mean Cppcheck CWE': round(s['cppcheck_cwe_count'].mean(), 3),
                'Mean Clang Total': round(s['clang_total'].mean(), 3),
                'Mean Clang Bugprone': round(s['clang_bugprone'].mean(), 3),
                'Mean Clang Analyzer': round(s['clang_analyzer'].mean(), 3),
            })

q12_status = pd.DataFrame(status_comparison)
q12_status.to_csv(f'{TABLES}/q12_warnings_by_pass_status.csv', index=False)
print('Table Q12c: Mean Warning Counts by Test Pass Status')
q12_status

In [ ]:
# Figure Q12: Violin plots of warning counts by pass status
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, model in enumerate(['gemma', 'llama', 'qwen', 'human']):
    ax = axes[idx // 2, idx % 2]
    m = tested[tested['model'] == model].copy()
    m['pass_status'] = m['pass_rate'].apply(
        lambda x: 'All Passed' if x == 1.0 else ('Partial' if x > 0 else 'None Passed')
    )

    data_by_status = []
    labels = []
    for status in ['All Passed', 'Partial', 'None Passed']:
        d = m[m['pass_status'] == status]['clang_total'].dropna()
        if len(d) > 0:
            data_by_status.append(d.values)
            labels.append(f'{status}\n(n={len(d)})')

    if data_by_status:
        vp = ax.violinplot(data_by_status, showmeans=True, showmedians=True)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Clang-Tidy Total Diagnostics')
    ax.set_title(f'{model.capitalize()}')

plt.suptitle('Clang-Tidy Diagnostics by Test Pass Status', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES}/q12_warnings_by_status_violin.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q12_warnings_by_status_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure Q12b: Scatter plot of pass_rate vs clang_total (AI models only)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, model in enumerate(['gemma', 'llama', 'qwen']):
    ax = axes[idx]
    m = tested[(tested['model'] == model)][['pass_rate', 'clang_total']].dropna()
    ax.scatter(m['pass_rate'], m['clang_total'], alpha=0.2, s=10, color='steelblue')

    # Add trend line
    z = np.polyfit(m['pass_rate'], m['clang_total'], 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(0, 1, 100)
    ax.plot(x_line, p_line(x_line), 'r-', alpha=0.7, linewidth=2)

    r, p = stats.spearmanr(m['pass_rate'], m['clang_total'])
    ax.set_xlabel('Test Pass Rate')
    ax.set_ylabel('Clang-Tidy Total Diagnostics')
    ax.set_title(f'{model.capitalize()} (rho={r:.3f}, p={p:.2e})')

plt.suptitle('Test Pass Rate vs Static Analysis Warnings', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES}/q12_tpr_vs_clang.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES}/q12_tpr_vs_clang.png', bbox_inches='tight')
plt.show()

---
## Q3: Investigation of gen_2 TPR Drops

> "The gen_2 TPR drops (e.g., Gemma 0.040) are extreme. Did you identify any pipeline bug, environment change, or prompt alteration that explains this?"

In [ ]:
# Investigate gen_2 behavior for all models
# Note: The paper reports Gemma TPR ~0.427 for gen_2, not 0.040.
# The reviewer may be referring to a specific tag or subset. Let's check.

# Overall TPR by model/generation
tpr_summary = tested.groupby(['model', 'generation']).agg(
    n=('pass_rate', 'count'),
    mean_tpr=('pass_rate', 'mean'),
    median_tpr=('pass_rate', 'median'),
    std_tpr=('pass_rate', 'std'),
    pct_zero=('pass_rate', lambda x: (x == 0).mean()),
    pct_all_pass=('pass_rate', lambda x: (x == 1.0).mean()),
).reset_index().round(3)

print('TPR Summary by Model/Generation:')
tpr_summary

In [ ]:
# Check if gen_2 has different characteristics: compile rate, code length, etc.
gen_comparison = tests.groupby(['model', 'generation']).agg(
    n=('sample_id', 'count'),
    compile_rate=('compiled', 'mean'),
    test_coverage=('has_tests', 'mean'),
    timeout_rate=('timeout', 'mean'),
    mean_total_tests=('total_tests', lambda x: x[x > 0].mean()),
).reset_index().round(3)

gen_comparison.to_csv(f'{TABLES}/q3_gen_comparison.csv', index=False)
print('Table Q3: Generation-Level Comparison')
gen_comparison

---
## Summary of All Generated Outputs

In [ ]:
import glob

# List all Q-prefixed outputs
tables = sorted(glob.glob(f'{TABLES}/q*'))
figures = sorted(glob.glob(f'{FIGURES}/q*'))

print('=== Generated Tables ===')
for t in tables:
    print(f'  {os.path.basename(t)}')

print(f'\n=== Generated Figures ===')
for f in figures:
    print(f'  {os.path.basename(f)}')